# Pipeline de modélisation — Dataset Titanic

> **Objectif :** Entraîner et comparer plusieurs modèles de classification sur les trois sélections de features issues de `Titanic_select.ipynb` (P, VT, KB), afin d'identifier la meilleure combinaison **modèle × sélection**.
**Auteurs :** Nickels Ethan
---

## Table des matières

1. [Importation des librairies et chargement des données](#section-1)
2. [Définition des modèles et de la fonction d'évaluation](#section-2)
3. [Entraînement sur la sélection Personnelle (P)](#section-3)
4. [Entraînement sur Variance Threshold (VT)](#section-4)
5. [Entraînement sur Select K-Best (KB)](#section-5)
6. [Comparaison globale des résultats](#section-6)
7. [Analyse du meilleur modèle](#section-7)
8. [Validation croisée du meilleur modèle](#section-8)
9. [Conclusion](#section-9)


---
## 1. Importation des librairies et chargement des données <a id='section-1'></a>

On importe les modèles de classification à comparer ainsi que les métriques d'évaluation. On charge ensuite les 8 fichiers CSV exportés par `Titanic_select.ipynb` (3 sélections de features + cibles), qui doivent se trouver dans le même répertoire que ce notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score, StratifiedKFold

RANDOM_STATE = 42
pd.pandas.set_option('display.max_columns', None)


In [ ]:
# Chargement des cibles (identiques pour les 3 sélections)
y_train = pd.read_csv('ytrain.csv').squeeze()
y_test  = pd.read_csv('ytest.csv').squeeze()

# Chargement des 3 sélections de features
selections = {
    'P': {
        'X_train': pd.read_csv('xtrain_P.csv'),
        'X_test':  pd.read_csv('xtest_P.csv'),
    },
    'VT': {
        'X_train': pd.read_csv('xtrain_VT.csv'),
        'X_test':  pd.read_csv('xtest_VT.csv'),
    },
    'KB': {
        'X_train': pd.read_csv('xtrain_KB.csv'),
        'X_test':  pd.read_csv('xtest_KB.csv'),
    },
}

for name, data in selections.items():
    print(f"Sélection {name} : X_train {data['X_train'].shape} | X_test {data['X_test'].shape} | features = {data['X_train'].columns.tolist()}")


Les 3 sélections sont chargées avec les mêmes cibles (`y_train`, `y_test`). Chaque sélection contient un nombre de features différent (6 pour P et KB, 8 pour VT), ce qui va permettre de comparer leur pouvoir prédictif respectif.

---
## 2. Définition des modèles et de la fonction d'évaluation <a id='section-2'></a>

On compare **5 modèles de classification**, couvrant des familles d'algorithmes différentes :
- **Régression Logistique** : modèle linéaire, simple et interprétable, bonne baseline.
- **KNN** : basé sur la distance, sensible à la normalisation (déjà appliquée en preprocessing).
- **Arbre de décision** : modèle non-linéaire, facilement interprétable.
- **Random Forest** : ensemble d'arbres, robuste au surapprentissage.
- **SVM (noyau RBF)** : efficace sur des frontières de décision non-linéaires.

Pour chaque combinaison (sélection × modèle), on calcule 4 métriques : **accuracy**, **precision**, **recall** et **f1-score**. Le f1-score est particulièrement pertinent ici car les classes (survécu / non-survécu) sont légèrement déséquilibrées (~38 % / 62 %).


In [ ]:
# Dictionnaire des modèles à comparer
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200),
    'SVM (RBF)':           SVC(kernel='rbf', random_state=RANDOM_STATE),
}


def evaluate_model(model, X_train, y_train, X_test, y_test):
    """Entraîne un modèle et retourne ses métriques d'évaluation sur le test set."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
    }


def run_pipeline(selection_name, X_train, y_train, X_test, y_test, models):
    """Entraîne et évalue tous les modèles sur une sélection de features donnée."""
    results = []
    for model_name, model in models.items():
        metrics = evaluate_model(model, X_train, y_train, X_test, y_test)
        metrics['selection'] = selection_name
        metrics['model'] = model_name
        results.append(metrics)
    return pd.DataFrame(results)[['selection', 'model', 'accuracy', 'precision', 'recall', 'f1']]


Ces deux fonctions constituent le cœur du pipeline : `evaluate_model` entraîne un modèle unique, `run_pipeline` l'applique à tous les modèles pour une sélection donnée. Cette architecture permet d'exécuter le même pipeline sur P, VT et KB sans dupliquer de code.

---
## 3. Entraînement sur la sélection Personnelle (P) <a id='section-3'></a>

In [ ]:
results_P = run_pipeline('P', selections['P']['X_train'], y_train, selections['P']['X_test'], y_test, models)
results_P.sort_values('f1', ascending=False)


---
## 4. Entraînement sur Variance Threshold (VT) <a id='section-4'></a>

In [ ]:
results_VT = run_pipeline('VT', selections['VT']['X_train'], y_train, selections['VT']['X_test'], y_test, models)
results_VT.sort_values('f1', ascending=False)


---
## 5. Entraînement sur Select K-Best (KB) <a id='section-5'></a>

In [ ]:
results_KB = run_pipeline('KB', selections['KB']['X_train'], y_train, selections['KB']['X_test'], y_test, models)
results_KB.sort_values('f1', ascending=False)


---
## 6. Comparaison globale des résultats <a id='section-6'></a>

On rassemble les résultats des 3 sélections (5 modèles × 3 sélections = 15 combinaisons) dans un seul tableau, puis on les visualise pour identifier la meilleure combinaison.


In [ ]:
# Fusion des résultats des 3 sélections
all_results = pd.concat([results_P, results_VT, results_KB], ignore_index=True)
all_results_sorted = all_results.sort_values('f1', ascending=False).reset_index(drop=True)
all_results_sorted


In [ ]:
# Visualisation comparative (f1-score par modèle et par sélection)
plt.figure(figsize=(12, 6))
sns.barplot(data=all_results, x='model', y='f1', hue='selection', palette='viridis')
plt.title("Comparaison du f1-score par modèle et par sélection de features")
plt.ylabel('F1-score')
plt.xlabel('Modèle')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1)
plt.legend(title='Sélection')
plt.tight_layout()
plt.show()


Le graphique permet de comparer visuellement les 15 combinaisons. On observe si certaines sélections avantagent systématiquement certains modèles, ou si un modèle domine quelle que soit la sélection utilisée.

---
## 7. Analyse du meilleur modèle <a id='section-7'></a>

On identifie automatiquement la meilleure combinaison (sélection, modèle) selon le f1-score, puis on affiche sa matrice de confusion pour analyser en détail ses erreurs.


In [ ]:
# Identification de la meilleure combinaison
best_row = all_results_sorted.iloc[0]
best_selection = best_row['selection']
best_model_name = best_row['model']

print(f"Meilleure combinaison : {best_model_name} sur la sélection {best_selection}")
print(best_row[['accuracy', 'precision', 'recall', 'f1']])

# Ré-entraînement du meilleur modèle (nouvelle instance propre)
best_model = models[best_model_name].__class__(**models[best_model_name].get_params())
X_train_best = selections[best_selection]['X_train']
X_test_best  = selections[best_selection]['X_test']

best_model.fit(X_train_best, y_train)
y_pred_best = best_model.predict(X_test_best)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-survécu', 'Survécu'])
disp.plot(cmap='Blues', values_format='d')
plt.title(f"Matrice de confusion — {best_model_name} ({best_selection})")
plt.show()


La matrice de confusion permet de voir précisément le type d'erreurs commises : faux positifs (prédits survivants mais morts) vs faux négatifs (prédits morts mais survivants). Dans un contexte comme celui-ci, ces deux types d'erreurs n'ont pas nécessairement le même coût selon l'usage du modèle.

---
## 8. Validation croisée du meilleur modèle <a id='section-8'></a>

Le score obtenu sur un seul split train/test peut être optimiste ou pessimiste selon le hasard du découpage. On confirme la robustesse du meilleur modèle avec une **validation croisée stratifiée à 5 plis** (5-fold), réalisée sur l'ensemble des données d'entraînement.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_model, X_train_best, y_train, cv=cv, scoring='f1')

print(f"Scores f1 sur les 5 plis : {np.round(cv_scores, 3)}")
print(f"F1-score moyen (CV)     : {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
print(f"F1-score sur le test set : {best_row['f1']:.3f}")


Si le f1-score moyen en validation croisée est proche du f1-score obtenu sur le test set, cela confirme que le modèle généralise bien et que sa performance n'est pas due au hasard du split initial. Un écart important indiquerait un possible surapprentissage ou un split test non représentatif.

---
## 9. Conclusion <a id='section-9'></a>

Ce pipeline a permis de tester **5 modèles de classification** sur **3 sélections de features** différentes (Personnelle, Variance Threshold, Select K-Best), soit 15 combinaisons évaluées de façon systématique et reproductible.

**Résumé de la démarche complète du projet :**
1. `Titanic_Analyse.ipynb` : analyse exploratoire, identification des variables clés (`pclass`, `sex`, `fare`, titre social).
2. `Titanic_preprocessing.ipynb` : nettoyage, imputation, encodage, transformation log1p, normalisation, split train/test.
3. `Titanic_select.ipynb` : 3 méthodes de sélection de features (P, VT, KB) comparées et exportées.
4. `Titanic_pipeline.ipynb` (ce notebook) : entraînement, comparaison et validation du meilleur modèle.

La meilleure combinaison identifiée ci-dessus constitue le modèle final recommandé pour la prédiction de la survie sur le dataset Titanic.
